# Entraîner et évaluer des modèles sur les avis sur des médicaments avec Spark

Ce TP fait suite au TP de préparation des avis sur des médicaments : il en reprend le nettoyage et le pipeline de préparation, déjà écrits, et passe à l'apprentissage. Vous entraînez une régression logistique qui prédit si un avis est positif, l'évaluez, la régularisez, interprétez ses coefficients, choisissez son seuil de décision puis ses hyperparamètres par validation croisée, la comparez à une forêt aléatoire, et sauvegardez le pipeline complet. Vous finissez par deux méthodes sans étiquette : regrouper les mots des avis par k-means, et chercher des règles d'association par FP-Growth.

Chaque exercice s'appuie sur le précédent : faites-les dans l'ordre. Spark tourne ici en mode local, sur les deux cœurs de la machine de Colab : certains entraînements prennent une à deux minutes. Les diapositives travaillent sur la seule partie test du jeu de données, coupée en deux ; ce TP entraîne sur la partie entraînement et évalue sur la partie test : vos chiffres seront donc différents.

## Installation et données

Les données sont le [Drug Review Dataset](https://archive.ics.uci.edu/dataset/462/drug+review+dataset+drugs+com) (Gräßer et Kallumadi, UCI Machine Learning Repository, 2018, licence CC BY 4.0) : des avis publiés sur Drugs.com de 2008 à 2017, en anglais. Ses auteurs l'ont coupé en deux fichiers : `drugsComTrain_raw.tsv` (84 Mo), la partie entraînement, et `drugsComTest_raw.tsv` (28 Mo), la partie test. Chaque ligne est un avis : le médicament (`drugName`), l'affection soignée (`condition`), le texte (`review`), la note de 1 à 10 (`rating`), la date (`date`) et le nombre de lecteurs qui ont trouvé l'avis utile (`usefulCount`).

In [ ]:
!pip install pyspark
!wget -q -nc https://github.com/shuuchuu/datasets/raw/refs/heads/main/drug-reviews/drugsComTrain_raw.tsv
!wget -q -nc https://github.com/shuuchuu/datasets/raw/refs/heads/main/drug-reviews/drugsComTest_raw.tsv

In [ ]:
import html
import os
import shutil
import sys
import time

import numpy as np
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import (
  BinaryClassificationEvaluator,
  ClusteringEvaluator,
  MulticlassClassificationEvaluator,
)
from pyspark.ml.feature import (
  IDF,
  Binarizer,
  CountVectorizer,
  OneHotEncoder,
  RegexTokenizer,
  SQLTransformer,
  StandardScaler,
  StopWordsRemover,
  StringIndexer,
  VectorAssembler,
  Word2Vec,
)
from pyspark.ml.fpm import FPGrowth
from pyspark.ml.functions import vector_to_array
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

La session tourne en local sur deux cœurs, avec 4 Go de mémoire pour le driver (1 Go par défaut, trop peu pour la validation croisée), et quatre partitions pour les agrégations au lieu des 200 par défaut. Les barres de progression de Spark sont masquées : elles encombreraient les sorties. Les processus Python que Spark lance pour exécuter la fonction Python `decoder` du nettoyage prennent l'interpréteur de ce notebook, `sys.executable` : hors de Colab, Spark prendrait sinon le `python3` du `PATH`, qui peut être d'une autre version de Python que celui du driver, ce que Spark refuse.

In [ ]:
os.environ["PYSPARK_PYTHON"] = sys.executable
spark = (
  SparkSession.builder.appName("avis-apprentissage")
  .master("local[2]")
  .config("spark.driver.memory", "4g")
  .config("spark.sql.shuffle.partitions", "4")
  .config("spark.ui.showConsoleProgress", "false")
  .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

## Le nettoyage et la préparation, repris du TP précédent

Ces fonctions reprennent la solution du TP de préparation. `lire` charge un fichier d'avis (un avis peut tenir sur plusieurs lignes) et le répartit sur quatre partitions. `nettoyer` retire les doublons (un même avis publié sous le nom commercial et sous le nom de la molécule), remplace les conditions invalides et manquantes par `"inconnue"`, décode le texte, convertit la date dans `jour` et ajoute l'étiquette `positif`, 1 pour une note de 7 ou plus. `preparation` rend un pipeline neuf, pas encore entraîné : trois variables numériques (logarithme de `usefulCount`, longueur de l'avis, année) centrées et réduites, la condition encodée en one-hot et le TF-IDF des mots de l'avis, réunis dans la colonne `features`.

In [ ]:
cles = ["review", "date", "rating", "usefulCount"]
invalide = F.col("condition").contains("</span>") | F.col("condition").rlike("^[a-z]")
decoder = F.udf(lambda texte: html.unescape(texte[1:-1]), "string")
mots = RegexTokenizer(
  inputCol="review", outputCol="mots", pattern=r"[^\w']+", minTokenLength=2
)
utiles = StopWordsRemover(inputCol="mots", outputCol="mots_utiles")


def lire(fichier):
  return (
    spark.read.csv(
      fichier,
      sep="\t",
      header=True,
      inferSchema=True,
      multiLine=True,
      escape='"',
    )
    .withColumnRenamed("_c0", "id")
    .repartition(4)
  )


def nettoyer(avis):
  propres = (
    avis.dropDuplicates(cles)
    .withColumn("condition", F.when(~invalide, F.col("condition")))
    .withColumn("review", decoder("review"))
    .withColumn("jour", F.to_date("date", "MMMM d, yyyy"))
    .fillna({"condition": "inconnue"})
  )
  return Binarizer(threshold=6.0, inputCol="rating", outputCol="positif").transform(
    propres
  )


def preparation():
  return Pipeline(
    stages=[
      SQLTransformer(
        statement="""
          SELECT *, log1p(usefulCount) AS log_utile, length(review) AS longueur,
                 year(jour) AS annee
          FROM __THIS__
        """
      ),
      StringIndexer(
        inputCol="condition", outputCol="condition_idx", handleInvalid="keep"
      ),
      OneHotEncoder(inputCol="condition_idx", outputCol="condition_vec"),
      mots,
      utiles,
      CountVectorizer(inputCol="mots_utiles", outputCol="tf", vocabSize=10000, minDF=5),
      IDF(inputCol="tf", outputCol="tfidf"),
      VectorAssembler(
        inputCols=["log_utile", "longueur", "annee"], outputCol="numeriques"
      ),
      StandardScaler(inputCol="numeriques", outputCol="numeriques_std", withMean=True),
      VectorAssembler(
        inputCols=["numeriques_std", "condition_vec", "tfidf"], outputCol="features"
      ),
    ]
  )

## 1. Les deux parties du jeu

Lisez et nettoyez `drugsComTrain_raw.tsv` dans `entrainement` et `drugsComTest_raw.tsv` dans `test_complet`, avec `lire` et `nettoyer`, et mettez-les en cache. Combien d'avis compte chacun ?

Le nettoyage retire les doublons à l'intérieur d'un fichier, pas d'un fichier à l'autre. Gardez dans `communs` les avis de `test_complet` dont le texte, la date, la note et `usefulCount` (la liste `cles`) se retrouvent dans `entrainement`, et dans `test` les autres, en cache. Combien y en a-t-il ? Affichez cinq avis communs, avec le nom du médicament dans chacune des deux parties. Pourquoi évaluer le modèle sur `test` plutôt que sur `test_complet` ?


**Documentation utile**

- [pyspark.sql.DataFrame.join](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.join.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Une jointure `"left_semi"` garde les lignes de gauche qui ont une correspondance à droite, sans en ajouter les colonnes ; `"left_anti"`, celles qui n'en ont pas. Joignez sur `entrainement.select(cles)`, pour ne comparer que les clés. Pour afficher les deux noms, faites une jointure ordinaire sur `cles`, après avoir renommé le `drugName` de `entrainement`.

```python
entrainement = nettoyer(lire(...)).cache()
test_complet = nettoyer(lire(...)).cache()
print(entrainement.count(), test_complet.count())
communs = test_complet.join(entrainement.select(cles), cles, ...)
test = test_complet.join(entrainement.select(cles), cles, ...).cache()
print(communs.count(), test.count())
noms = entrainement.select(cles + ["drugName"]).withColumnRenamed(
  "drugName", "nom_entrainement"
)
test_complet.join(noms, cles).select(
  "drugName", "nom_entrainement", "rating", "date"
).show(5)
```

### Solution

In [ ]:
entrainement = nettoyer(lire("drugsComTrain_raw.tsv")).cache()
test_complet = nettoyer(lire("drugsComTest_raw.tsv")).cache()
print(entrainement.count(), test_complet.count())
communs = test_complet.join(entrainement.select(cles), cles, "left_semi")
test = test_complet.join(entrainement.select(cles), cles, "left_anti").cache()
print(communs.count(), test.count())
noms = entrainement.select(cles + ["drugName"]).withColumnRenamed(
  "drugName", "nom_entrainement"
)
test_complet.join(noms, cles).select(
  "drugName", "nom_entrainement", "rating", "date"
).show(5)
# 112 626 avis d'entraînement et 48 352 de test, dont 32 138 (deux sur trois) déjà
# dans l'entraînement, sous l'autre nom du médicament (Xanax et Alprazolam). Un modèle
# évalué sur test_complet le serait surtout sur des avis qu'il a appris : son score
# serait trop optimiste. Il reste 16 214 avis de test vraiment nouveaux.

## 2. Préparer les variables

Entraînez le pipeline de `preparation()` sur `entrainement`, dans `modele_preparation`, et appliquez-le à `entrainement`, à `test` et à `communs`. Gardez les colonnes `features` et `positif` dans `entrainement_prep`, `test_prep` et `communs_prep`, en cache.

Quelle est la taille des vecteurs `features` ? Quelle part des avis de `test` est positive ? C'est l'exactitude d'un modèle qui répondrait toujours « positif », la référence à battre.

In [ ]:
# Votre code ici

### Conseils supplémentaires

Le pipeline s'entraîne sur l'entraînement seulement : le vocabulaire, les poids IDF, les catégories et les moyennes ne voient pas le test. Un vecteur a un attribut `size`.

```python
modele_preparation = preparation().fit(...)
colonnes = ["features", "positif"]
entrainement_prep = modele_preparation.transform(...).select(colonnes).cache()
test_prep = ...
communs_prep = ...
print(entrainement_prep.first()["features"].size)
print(test_prep.filter(...).count() / test_prep.count())
```

### Solution

In [ ]:
modele_preparation = preparation().fit(entrainement)
colonnes = ["features", "positif"]
entrainement_prep = modele_preparation.transform(entrainement).select(colonnes).cache()
test_prep = modele_preparation.transform(test).select(colonnes).cache()
communs_prep = modele_preparation.transform(communs).select(colonnes).cache()
print(entrainement_prep.first()["features"].size)
print(test_prep.filter("positif = 1").count() / test_prep.count())
# 10 775 dimensions : 3 variables numériques, 772 pour la condition et 10 000 mots.
# 66 % des avis du test sont positifs : un modèle qui répondrait toujours « positif »
# aurait une exactitude de 0,665.

## 3. Une première régression logistique

Entraînez sur `entrainement_prep`, dans `modele`, une régression logistique sans régularisation (`LogisticRegression`, étiquette `positif`, 100 itérations au plus), en mesurant la durée de l'entraînement. Son résumé (`summary`) donne le nombre d'itérations faites, la perte à chaque itération et l'AUC sur les données d'entraînement : affichez-les. Le modèle a-t-il convergé ?

Affichez enfin `positif`, `probability` et `prediction` pour cinq avis du test. Comment la prédiction se déduit-elle de la probabilité ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

Sans régularisation, c'est le `regParam=0.0` par défaut. Le résumé a les attributs `totalIterations`, `objectiveHistory` (une liste) et `areaUnderROC`. `time.time()` donne l'heure en secondes.

```python
debut = time.time()
modele = LogisticRegression(labelCol=..., maxIter=...).fit(...)
print(time.time() - debut)
resume = modele.summary
print(resume.totalIterations, resume.objectiveHistory[:3], resume.objectiveHistory[-1])
print(resume.areaUnderROC)
modele.transform(test_prep).select("positif", "probability", "prediction").show(
  5, truncate=False
)
```

### Solution

In [ ]:
debut = time.time()
modele = LogisticRegression(labelCol="positif", maxIter=100).fit(entrainement_prep)
print(time.time() - debut)
resume = modele.summary
print(resume.totalIterations, resume.objectiveHistory[:3], resume.objectiveHistory[-1])
print(resume.areaUnderROC)
modele.transform(test_prep).select("positif", "probability", "prediction").show(
  5, truncate=False
)
# 100 itérations, le maximum : le modèle n'a pas convergé. La perte passe de 0,64 à
# 0,32, et l'AUC d'entraînement atteint 0,93. Un avis est prédit positif (1.0) quand
# la probabilité de la classe 1, le second élément de probability, dépasse 0,5.

## 4. Évaluer sur le test

Écrivez une fonction `evaluer(modele, donnees)` qui affiche l'aire sous la courbe ROC (`BinaryClassificationEvaluator`) et l'exactitude (`MulticlassClassificationEvaluator`) d'un modèle sur des données préparées. Appliquez-la à `modele` sur `test_prep`, puis affichez la matrice de confusion du test : le nombre d'avis pour chaque couple d'étiquette et de prédiction.

Comparez l'AUC du test à celle de l'entraînement, et l'exactitude à la référence de l'exercice 2 : qu'en concluez-vous ? Évaluez enfin `modele` sur `communs_prep` : pourquoi le score y est-il meilleur ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

Un évaluateur prend la colonne de l'étiquette (`labelCol`) et la mesure (`metricName`) : `areaUnderROC` est celle de `BinaryClassificationEvaluator` par défaut, `"accuracy"` celle qu'il faut demander à `MulticlassClassificationEvaluator`. Sa méthode `evaluate` prend les prédictions.

```python
auc = BinaryClassificationEvaluator(labelCol="positif")
exactitude = MulticlassClassificationEvaluator(labelCol=..., metricName=...)


def evaluer(modele, donnees):
  predictions = modele.transform(donnees)
  print(f"AUC {auc.evaluate(predictions):.3f}, exactitude {...:.3f}")


evaluer(modele, test_prep)
modele.transform(test_prep).groupBy(..., ...).count().orderBy(...).show()
evaluer(modele, communs_prep)
```

### Solution

In [ ]:
auc = BinaryClassificationEvaluator(labelCol="positif")
exactitude = MulticlassClassificationEvaluator(
  labelCol="positif", metricName="accuracy"
)


def evaluer(modele, donnees):
  predictions = modele.transform(donnees)
  print(
    f"AUC {auc.evaluate(predictions):.3f}, "
    f"exactitude {exactitude.evaluate(predictions):.3f}"
  )


evaluer(modele, test_prep)
modele.transform(test_prep).groupBy("positif", "prediction").count().orderBy(
  "positif", "prediction"
).show()
evaluer(modele, communs_prep)
# Test : AUC 0,855 et exactitude 0,803, contre 0,927 pour l'AUC d'entraînement : le
# modèle surapprend. Il bat la référence (0,665), mais rate 1 669 avis négatifs sur
# 5 437. Sur les avis communs, il retrouve son score d'entraînement (AUC 0,927) : il
# les a vus, sous l'autre nom du médicament. C'est le score qu'aurait donné
# test_complet, trop optimiste.

## 5. Régulariser

Entraînez quatre régressions logistiques régularisées : en L2 (`elasticNetParam=0`), de force `regParam` 0,01 puis 0,1, et en L1 (`elasticNetParam=1`), de force 0,001 puis 0,003. Pour chacune, affichez l'AUC d'entraînement, l'AUC du test et le nombre de coefficients non nuls. Laquelle surapprend le moins ? Que fait la régularisation L1 aux coefficients ? Gardez le modèle L1 de force 0,003 dans `modele_l1`.

Si vous avez fait le TP de suivi des expériences avec MLflow, vous y avez déjà comparé trois valeurs de `regParam` par leur AUC de test et leur convergence ; cet exercice regarde ce que la régularisation fait au surapprentissage (l'écart entre l'AUC d'entraînement et celle du test) et aux coefficients, en L2 puis en L1.

Comparer ces modèles sur le test sert ici à voir l'effet de la régularisation ; l'exercice 8 choisira les hyperparamètres sans le regarder.


**Documentation utile**

- [numpy.count_nonzero](https://numpy.org/doc/stable/reference/generated/numpy.count_nonzero.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Une boucle sur les couples `(regParam, elasticNetParam)`. Les coefficients sont un vecteur, `modele.coefficients` : `toArray()` en fait un tableau NumPy, dont `np.count_nonzero` compte les valeurs non nulles.

```python
for force, melange in [(0.01, 0.0), (0.1, 0.0), (..., 1.0), (..., 1.0)]:
  regularise = LogisticRegression(
    labelCol="positif", regParam=..., elasticNetParam=...
  ).fit(entrainement_prep)
  print(
    force,
    melange,
    f"{regularise.summary.areaUnderROC:.3f}",
    f"{auc.evaluate(regularise.transform(test_prep)):.3f}",
    np.count_nonzero(regularise.coefficients.toArray()),
  )
  if (force, melange) == (0.003, 1.0):
    modele_l1 = regularise
```

### Solution

In [ ]:
for force, melange in [(0.01, 0.0), (0.1, 0.0), (0.001, 1.0), (0.003, 1.0)]:
  regularise = LogisticRegression(
    labelCol="positif", regParam=force, elasticNetParam=melange
  ).fit(entrainement_prep)
  print(
    force,
    melange,
    f"{regularise.summary.areaUnderROC:.3f}",
    f"{auc.evaluate(regularise.transform(test_prep)):.3f}",
    np.count_nonzero(regularise.coefficients.toArray()),
  )
  if (force, melange) == (0.003, 1.0):
    modele_l1 = regularise
# AUC d'entraînement et de test : 0,926 et 0,865 (L2, 0,01), 0,917 et 0,875 (L2,
# 0,1), 0,910 et 0,880 (L1, 0,001), 0,881 et 0,873 (L1, 0,003). Plus la
# régularisation est forte, plus l'écart entre entraînement et test se réduit. La
# régularisation L1 met des coefficients à zéro : 4 777, puis 1 076 non nuls sur
# 10 775, une sélection des variables utiles.

## 6. Interpréter les coefficients

Les coefficients de `modele_l1` suivent l'ordre des variables de `features` : les trois variables numériques, puis l'encodage de la condition, puis les mots du vocabulaire du `CountVectorizer`. Retrouvez la taille de l'encodage de la condition dans `modele_preparation`, puis affichez les dix mots aux coefficients les plus négatifs, les dix aux coefficients les plus positifs, et les coefficients des trois variables numériques. Que disent-ils des avis positifs ?


**Documentation utile**

- [numpy.argsort](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`modele_preparation.stages` est la liste des étages entraînés, dans l'ordre du pipeline : l'encodeur `OneHotEncoderModel` est le troisième (`stages[2]`), le `CountVectorizerModel` le sixième (`stages[5]`). L'encodeur a appris le nombre de catégories, `categorySizes[0]` : les conditions, plus une dernière, celle des conditions inconnues, qu'ajoute `handleInvalid="keep"` du `StringIndexer`. C'est cette dernière qu'il retire du vecteur (`dropLast=True`), d'où le `- 1`. Le `CountVectorizerModel` a un attribut `vocabulary`. `np.argsort` rend les indices qui trient un tableau.

```python
taille_condition = modele_preparation.stages[2].categorySizes[0] - 1
vocabulaire = modele_preparation.stages[5].vocabulary
coefficients = modele_l1.coefficients.toArray()
poids_mots = coefficients[3 + ... :]
ordre = np.argsort(poids_mots)
print([(vocabulaire[i], round(poids_mots[i], 2)) for i in ordre[:10]])
print([(vocabulaire[i], round(poids_mots[i], 2)) for i in ordre[::-1][:10]])
print(dict(zip(["log_utile", "longueur", "annee"], coefficients[:3].round(2))))
```

### Solution

In [ ]:
taille_condition = modele_preparation.stages[2].categorySizes[0] - 1
vocabulaire = modele_preparation.stages[5].vocabulary
coefficients = modele_l1.coefficients.toArray()
poids_mots = coefficients[3 + taille_condition :]
ordre = np.argsort(poids_mots)
print([(vocabulaire[i], round(poids_mots[i], 2)) for i in ordre[:10]])
print([(vocabulaire[i], round(poids_mots[i], 2)) for i in ordre[::-1][:10]])
print(dict(zip(["log_utile", "longueur", "annee"], coefficients[:3].round(2))))
# Les mots les plus négatifs disent la déception (disappointed, worse, worst,
# ruined, waste, horrible), les plus positifs le soulagement (love, saved, miracle,
# amazing, best). Un avis jugé utile par beaucoup de lecteurs est plutôt positif
# (log_utile, +0,40), un avis récent plutôt négatif (annee, -0,17).

## 7. Choisir le seuil de décision

Un laboratoire veut repérer les avis négatifs sur ses médicaments, pour les faire relire. Pour les seuils 0,5, 0,6, 0,7 et 0,8 de `modele_l1` (`setThreshold`), affichez la part des avis négatifs du test que le modèle détecte (le rappel de la classe 0), la part des avis prédits négatifs qui le sont vraiment (sa précision) et l'exactitude. Quel seuil choisiriez-vous ? L'AUC change-t-elle avec le seuil ? Remettez le seuil à 0,5.

Comparer ces seuils sur le test sert ici à voir l'effet du seuil ; en pratique, il se choisit sur une partie de validation.

In [ ]:
# Votre code ici

### Conseils supplémentaires

`MulticlassClassificationEvaluator` mesure le rappel et la précision d'une seule classe : `metricName="recallByLabel"` ou `"precisionByLabel"`, avec `metricLabel=0.0`. Un avis est prédit positif quand sa probabilité dépasse le seuil.

```python
rappel_negatifs = MulticlassClassificationEvaluator(
  labelCol="positif", metricName=..., metricLabel=0.0
)
precision_negatifs = MulticlassClassificationEvaluator(
  labelCol="positif", metricName=..., metricLabel=0.0
)
for seuil in [0.5, 0.6, 0.7, 0.8]:
  predictions = modele_l1.setThreshold(...).transform(test_prep)
  print(
    seuil,
    f"{rappel_negatifs.evaluate(predictions):.3f}",
    f"{precision_negatifs.evaluate(predictions):.3f}",
    f"{exactitude.evaluate(predictions):.3f}",
    f"{auc.evaluate(predictions):.3f}",
  )
modele_l1.setThreshold(0.5)
```

### Solution

In [ ]:
rappel_negatifs = MulticlassClassificationEvaluator(
  labelCol="positif", metricName="recallByLabel", metricLabel=0.0
)
precision_negatifs = MulticlassClassificationEvaluator(
  labelCol="positif", metricName="precisionByLabel", metricLabel=0.0
)
for seuil in [0.5, 0.6, 0.7, 0.8]:
  predictions = modele_l1.setThreshold(seuil).transform(test_prep)
  print(
    seuil,
    f"{rappel_negatifs.evaluate(predictions):.3f}",
    f"{precision_negatifs.evaluate(predictions):.3f}",
    f"{exactitude.evaluate(predictions):.3f}",
    f"{auc.evaluate(predictions):.3f}",
  )
modele_l1.setThreshold(0.5)
# Au seuil de 0,5, le modèle trouve 63 % des avis négatifs, et 75 % de ceux qu'il
# signale le sont vraiment. Au seuil de 0,7, il en trouve 87 %, mais 60 % seulement
# des signalés sont négatifs : plus de relectures inutiles. Pour une relecture
# humaine, 0,6 ou 0,7 rattrapent bien plus d'avis négatifs pour un coût raisonnable.
# L'AUC (0,873) ne change pas : elle ne dépend que de l'ordre des probabilités.

## 8. Choisir les hyperparamètres par validation croisée

Cherchez les meilleurs hyperparamètres d'une régression logistique par validation croisée à trois plis sur `entrainement_prep`, avec l'AUC comme mesure : `regParam` parmi 0,001, 0,003 et 0,01, `elasticNetParam` parmi 0,5 et 1 (`ParamGridBuilder`, `CrossValidator`, deux modèles entraînés à la fois, graine 42). Mesurez sa durée. Combien de modèles entraîne-t-elle en tout ?

Affichez l'AUC moyenne de chaque combinaison (`avgMetrics`), les hyperparamètres retenus et l'AUC du meilleur modèle sur le test. Gardez-les dans `meilleure_force` et `meilleur_melange`. Le meilleur est-il au bord de la grille ? Quelles valeurs essayer ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

La grille se construit par `addGrid(estimateur.parametre, valeurs)`, une fois par paramètre, puis `build()`. `CrossValidator` prend l'estimateur, la grille (`estimatorParamMaps`), l'évaluateur, `numFolds`, `parallelism` et `seed` ; son modèle garde le meilleur modèle, réentraîné sur tout l'entraînement, dans `bestModel`.

```python
logistique = LogisticRegression(labelCol="positif")
grille = (
  ParamGridBuilder()
  .addGrid(logistique.regParam, [...])
  .addGrid(logistique.elasticNetParam, [...])
  .build()
)
validation = CrossValidator(
  estimator=logistique,
  estimatorParamMaps=grille,
  evaluator=auc,
  numFolds=...,
  parallelism=...,
  seed=42,
)
debut = time.time()
modele_cv = validation.fit(entrainement_prep)
print(time.time() - debut)
for parametres, moyenne in zip(grille, modele_cv.avgMetrics):
  print(
    parametres[logistique.regParam], parametres[logistique.elasticNetParam], moyenne
  )
meilleure_force = modele_cv.bestModel.getRegParam()
meilleur_melange = modele_cv.bestModel.getElasticNetParam()
print(meilleure_force, meilleur_melange)
evaluer(modele_cv.bestModel, test_prep)
```

### Solution

In [ ]:
logistique = LogisticRegression(labelCol="positif")
grille = (
  ParamGridBuilder()
  .addGrid(logistique.regParam, [0.001, 0.003, 0.01])
  .addGrid(logistique.elasticNetParam, [0.5, 1.0])
  .build()
)
validation = CrossValidator(
  estimator=logistique,
  estimatorParamMaps=grille,
  evaluator=auc,
  numFolds=3,
  parallelism=2,
  seed=42,
)
debut = time.time()
modele_cv = validation.fit(entrainement_prep)
print(time.time() - debut)
for parametres, moyenne in zip(grille, modele_cv.avgMetrics):
  print(
    parametres[logistique.regParam], parametres[logistique.elasticNetParam], moyenne
  )
meilleure_force = modele_cv.bestModel.getRegParam()
meilleur_melange = modele_cv.bestModel.getElasticNetParam()
print(meilleure_force, meilleur_melange)
evaluer(modele_cv.bestModel, test_prep)
# 6 combinaisons × 3 plis = 18 modèles, plus le meilleur réentraîné sur tout
# l'entraînement : 19, en un peu plus de deux minutes. La validation croisée retient
# regParam = 0,003 et elasticNetParam = 0,5 (AUC moyenne de 0,879), qui donne sur le
# test une AUC de 0,880 et une exactitude de 0,815 : aussi bien que le meilleur de
# l'exercice 5 (L1 à 0,001), mais choisi cette fois sans regarder le test.
# elasticNetParam = 0,5 est au bord de la grille, sa plus petite valeur : à essayer,
# 0,25 et 0 (la L2 pure), avec des regParam plus forts, la L2 en demandant 10 à 100
# fois plus que la L1 (0,1 à l'exercice 5). regParam = 0,003 est au milieu de la
# grille : rien à étendre de ce côté.

## 9. Une forêt aléatoire

Entraînez une forêt aléatoire (`RandomForestClassifier`) de 30 arbres de profondeur 10 au plus, graine 42, en mesurant la durée de l'entraînement, et évaluez-la sur le test avec `evaluer`. Comparez son AUC à celle de la régression logistique de l'exercice 8, et sa durée à celle de l'entraînement de l'exercice 3.

Affichez les dix variables les plus importantes (`featureImportances`), en retrouvant le nom de chacune : une variable numérique, une condition ou un mot. Pourquoi la forêt fait-elle moins bien que le modèle linéaire sur ces données ?


**Documentation utile**

- [pyspark.ml.classification.RandomForestClassificationModel.featureImportances](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.classification.RandomForestClassificationModel.html#pyspark.ml.classification.RandomForestClassificationModel.featureImportances)

In [ ]:
# Votre code ici

### Conseils supplémentaires

L'importance d'une variable est un vecteur de la taille de `features`. Pour nommer une position, mettez bout à bout les noms des trois variables numériques, les conditions apprises par le `StringIndexerModel` (`modele_preparation.stages[1].labelsArray[0]`, dans l'ordre de leurs indices : toutes, la catégorie que l'encodeur retire étant celle des conditions inconnues, absente de la liste) et le vocabulaire.

```python
debut = time.time()
foret = RandomForestClassifier(
  labelCol="positif", numTrees=..., maxDepth=..., seed=42
).fit(entrainement_prep)
print(time.time() - debut)
evaluer(foret, test_prep)
conditions = modele_preparation.stages[1].labelsArray[0]
noms_variables = (
  ["log_utile", "longueur", "annee"]
  + [f"condition {c}" for c in conditions]
  + vocabulaire
)
importances = foret.featureImportances.toArray()
print(
  [(noms_variables[i], round(importances[i], 3)) for i in np.argsort(...)[::-1][:10]]
)
```

### Solution

In [ ]:
debut = time.time()
foret = RandomForestClassifier(
  labelCol="positif", numTrees=30, maxDepth=10, seed=42
).fit(entrainement_prep)
print(time.time() - debut)
evaluer(foret, test_prep)
conditions = modele_preparation.stages[1].labelsArray[0]
noms_variables = (
  ["log_utile", "longueur", "annee"]
  + [f"condition {c}" for c in conditions]
  + vocabulaire
)
importances = foret.featureImportances.toArray()
print(
  [
    (noms_variables[i], round(importances[i], 3))
    for i in np.argsort(importances)[::-1][:10]
  ]
)
# AUC 0,802 et exactitude 0,665, contre une AUC de 0,880 pour la régression
# logistique de l'exercice 8, et un entraînement environ deux fois plus long que les
# 100 itérations de l'exercice 3. L'exactitude est celle de la référence : la
# forêt répond presque toujours « positif ». Chaque nœud ne tire qu'une centaine de
# variables sur 10 775, surtout des mots rares, et des arbres de profondeur 10
# n'utilisent que quelques dizaines de mots : sur un texte creux en grande dimension,
# un modèle linéaire, qui pèse chaque mot, fait mieux. Les variables les plus
# importantes sont l'année, log_utile et des mots marqués (worst, works, saved).

## 10. Le pipeline complet

Réunissez dans un seul `Pipeline` les étages de `preparation()` et une régression logistique aux hyperparamètres retenus par l'exercice 8. Entraînez-le sur `entrainement`, les avis nettoyés mais pas encore préparés, sauvegardez-le dans le dossier `modele-avis` puis rechargez-le dans `modele_avis`.

Appliquez `modele_avis` à `test`, et affichez les cinq avis négatifs que le modèle croit positifs avec le plus d'assurance : leur note, la probabilité d'un avis positif et le début du texte. Pourquoi le modèle se trompe-t-il sur eux ?


**Documentation utile**

- [pyspark.ml.functions.vector_to_array](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.functions.vector_to_array.html)
- [pyspark.sql.functions.substring](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.substring.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`getStages()` rend la liste des étages d'un pipeline : ajoutez-y la régression logistique. La probabilité d'un avis positif est le second élément du vecteur `probability` : `vector_to_array` fait du vecteur un tableau dont on prend l'élément d'indice 1. `F.substring(colonne, 1, 150)` garde les 150 premiers caractères.

```python
complet = Pipeline(
  stages=preparation().getStages()
  + [LogisticRegression(labelCol="positif", regParam=..., elasticNetParam=...)]
)
shutil.rmtree("modele-avis", ignore_errors=True)
complet.fit(...).write().save("modele-avis")
modele_avis = PipelineModel.load(...)
(
  modele_avis.transform(test)
  .filter("positif = 0")
  .withColumn("proba_positif", vector_to_array("probability")[1])
  .orderBy(F.desc(...))
  .select("rating", "proba_positif", F.substring("review", 1, 150).alias("debut"))
  .show(5, truncate=False)
)
```

### Solution

In [ ]:
complet = Pipeline(
  stages=preparation().getStages()
  + [
    LogisticRegression(
      labelCol="positif", regParam=meilleure_force, elasticNetParam=meilleur_melange
    )
  ]
)
shutil.rmtree("modele-avis", ignore_errors=True)
complet.fit(entrainement).write().save("modele-avis")
modele_avis = PipelineModel.load("modele-avis")
(
  modele_avis.transform(test)
  .filter("positif = 0")
  .withColumn("proba_positif", vector_to_array("probability")[1])
  .orderBy(F.desc("proba_positif"))
  .select("rating", "proba_positif", F.substring("review", 1, 150).alias("debut"))
  .show(5, truncate=False)
)
# Les erreurs les plus sûres sont des avis mitigés (notes de 5 ou 6, juste sous le
# seuil de 7) ou des avis négatifs qui emploient des mots positifs : un médicament
# cité comme un « miracle sleep aid » qui a déçu, une dépendance (« I'm addicted »).
# Un sac de mots ne voit ni la négation, ni l'ironie, ni les guillemets.

## 11. Regrouper les mots par k-means

Le code suivant reprend les plongements de mots du TP de préparation. Il découpe en mots (les transformateurs `mots` et `utiles` de la préparation) un échantillon de 30 % des avis de `entrainement` (graine 42), dans `entrainement_mots`, en cache, et y entraîne un `Word2Vec` de `mots_utiles` vers `w2v` : des vecteurs de 50 dimensions (`vectorSize=50`), pour les mots présents au moins cinq fois (`minCount=5`). `getVectors()` rend le vecteur de chaque mot, un `DataFrame` de deux colonnes, `word` et `vector` : `mots_vecteurs` le garde en cache, la colonne du vecteur renommée en `features`, celle que lit `KMeans`.

In [ ]:
echantillon = entrainement.sample(fraction=0.3, seed=42)
entrainement_mots = utiles.transform(mots.transform(echantillon)).cache()
plongements = Word2Vec(
  inputCol="mots_utiles", outputCol="w2v", vectorSize=50, minCount=5, seed=42
).fit(entrainement_mots)
mots_vecteurs = plongements.getVectors().withColumnRenamed("vector", "features").cache()

Regroupez les mots de `mots_vecteurs` par k-means (`KMeans`, graine 42) pour $k$ de 5, 10 et 20 : affichez pour chacun la silhouette (`ClusteringEvaluator`) et le coût (`summary.trainingCost`). Pour $k = 10$, affichez les six mots les plus fréquents de chaque groupe, leur fréquence comptée dans `entrainement_mots` : quels thèmes y reconnaissez-vous ?


**Documentation utile**

- [pyspark.sql.functions.row_number](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.row_number.html)
- [pyspark.sql.functions.explode](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.explode.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

La fréquence d'un mot se compte en éclatant les listes de mots de `entrainement_mots` (`F.explode`), puis en groupant par mot. Pour garder les six mots les plus fréquents de chaque groupe, numérotez les mots de chaque groupe par fréquence décroissante avec une fenêtre (`F.row_number().over(...)`).

```python
silhouette = ClusteringEvaluator()
for k in [5, 10, 20]:
  kmeans = KMeans(k=k, seed=42).fit(mots_vecteurs)
  print(
    k, silhouette.evaluate(kmeans.transform(mots_vecteurs)), kmeans.summary.trainingCost
  )
frequences = (
  entrainement_mots.select(F.explode("mots_utiles").alias("word"))
  .groupBy("word")
  .count()
)
par_groupe = Window.partitionBy("prediction").orderBy(F.desc("count"))
(
  KMeans(k=10, seed=42)
  .fit(mots_vecteurs)
  .transform(mots_vecteurs.join(frequences, "word"))
  .withColumn("rang", F.row_number().over(par_groupe))
  .filter(...)
  .groupBy("prediction")
  .agg(F.collect_list("word").alias("mots"))
  .orderBy("prediction")
  .show(truncate=False)
)
```

### Solution

In [ ]:
silhouette = ClusteringEvaluator()
for k in [5, 10, 20]:
  kmeans = KMeans(k=k, seed=42).fit(mots_vecteurs)
  print(
    k, silhouette.evaluate(kmeans.transform(mots_vecteurs)), kmeans.summary.trainingCost
  )
frequences = (
  entrainement_mots.select(F.explode("mots_utiles").alias("word"))
  .groupBy("word")
  .count()
)
par_groupe = Window.partitionBy("prediction").orderBy(F.desc("count"))
(
  KMeans(k=10, seed=42)
  .fit(mots_vecteurs)
  .transform(mots_vecteurs.join(frequences, "word"))
  .withColumn("rang", F.row_number().over(par_groupe))
  .filter("rang <= 6")
  .groupBy("prediction")
  .agg(F.collect_list("word").alias("mots"))
  .orderBy("prediction")
  .show(truncate=False)
)
# Silhouettes de 0,41 (k = 5), 0,36 (k = 10) et 0,23 (k = 20) : la silhouette
# préfère peu de groupes, mais k = 10 en donne de plus lisibles. On y reconnaît le
# poids (weight, lost, gain, pounds), la peau (acne, skin, face), les douleurs et
# symptômes (pain, nausea, cramps), les doses (prescribed, mg, dose), le sommeil
# (sleep, night) et le cycle (pill, period).

## 12. Des règles d'association par FP-Growth

Le code suivant construit pour chaque avis de `entrainement_mots` un panier, la colonne `elements` de `paniers` : ses mots distincts (FP-Growth refuse un panier qui contient deux fois le même élément), plus l'élément `avis_positif` ou `avis_negatif` selon son étiquette. `FPGrowth` y cherche les ensembles d'éléments présents dans au moins 2 % des paniers (`minSupport`), et les règles de confiance 0,6 au moins (`minConfidence`). Le code affiche le nombre d'ensembles fréquents et de règles, les dix règles au lift le plus fort, puis les cinq règles de plus forte confiance qui concluent à `avis_positif`, et celles qui concluent à `avis_negatif`.

In [ ]:
etiquette = F.when(F.col("positif") == 1, F.lit("avis_positif")).otherwise(
  F.lit("avis_negatif")
)
paniers = entrainement_mots.select(
  F.array_distinct(F.concat("mots_utiles", F.array(etiquette))).alias("elements")
).cache()
motifs = FPGrowth(itemsCol="elements", minSupport=0.02, minConfidence=0.6).fit(paniers)
print(motifs.freqItemsets.count(), motifs.associationRules.count())
regles = motifs.associationRules
regles.orderBy(F.desc("lift")).show(10, truncate=False)
for conclusion in ["avis_positif", "avis_negatif"]:
  regles.filter(F.array_contains("consequent", conclusion)).orderBy(
    F.desc("confidence")
  ).show(5, truncate=False)

Que mesure le lift ? Pourquoi celui des règles qui concluent à `avis_positif` ne peut-il guère dépasser 1,5 ?

`motifs.transform` applique les règles à des paniers : sa colonne `prediction` reçoit les conséquents des règles dont l'antécédent est dans le panier. Construisez les paniers de `test`, sans l'élément d'étiquette, dans `paniers_test`, et appliquez-leur `motifs.transform` : quelle part des avis négatifs reçoit `avis_negatif` dans `prediction`, et quelle part des avis ainsi signalés est négative ? Comparez au rappel et à la précision de l'exercice 7.


**Documentation utile**

- [pyspark.sql.functions.array_contains](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.array_contains.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les paniers de `test` se construisent comme ceux de `paniers`, sans `etiquette` : `mots`, `utiles`, puis `F.array_distinct` sur `mots_utiles` ; gardez `positif`, pour compter. `F.array_contains(colonne, valeur)` teste si une liste contient une valeur : mettez-en le résultat dans une colonne `signale`. Le rappel est la part des avis négatifs signalés, la précision la part des avis signalés qui sont négatifs.

```python
paniers_test = utiles.transform(mots.transform(...)).select(
  "positif", F.array_distinct(...).alias("elements")
)
signales = motifs.transform(paniers_test).withColumn(
  "signale", F.array_contains("prediction", ...)
)
negatifs = signales.filter(...)
print(negatifs.filter("signale").count() / negatifs.count())
avis_signales = signales.filter(...)
print(avis_signales.filter(...).count() / avis_signales.count())
```

### Solution

In [ ]:
paniers_test = utiles.transform(mots.transform(test)).select(
  "positif", F.array_distinct("mots_utiles").alias("elements")
)
signales = motifs.transform(paniers_test).withColumn(
  "signale", F.array_contains("prediction", "avis_negatif")
)
negatifs = signales.filter("positif = 0")
print(negatifs.filter("signale").count() / negatifs.count())
avis_signales = signales.filter("signale")
print(avis_signales.filter("positif = 0").count() / avis_signales.count())
# Le lift divise la confiance par la part des paniers qui contiennent le conséquent :
# il mesure combien l'antécédent rend le conséquent plus probable que le hasard. Les
# plus forts relient des mots qui vont ensemble (panic attacks, blood pressure, mood
# swings, sex drive, birth control). avis_positif est dans 66 % des paniers : même une
# confiance de 1 ne donne qu'un lift de 1 / 0,662 ≈ 1,51.
# Une seule règle conclut à avis_negatif, worst : elle signale 7,6 % des avis négatifs
# du test (413 sur 5 437), et 63 % des avis qu'elle signale sont négatifs, contre un
# rappel de 63 % et une précision de 75 % pour la régression logistique de
# l'exercice 7, au seuil de 0,5. Une règle ne retient que des mots fréquents (2 % des
# paniers au moins) et assez sûrs (confiance de 0,6) : un seul mot conclut à
# avis_negatif, quand le modèle linéaire pèse les 10 000 mots du vocabulaire.

## Fin

Arrêtez la session, qui libère les exécuteurs.

In [ ]:
spark.stop()